In [11]:
import streamlit as st
import pandas as pd
import joblib
import os
import json
import matplotlib.pyplot as plt
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

st.set_page_config(
    page_title="Credit Card Fraud Detection",
    page_icon="💳",
    layout="wide"
)

MODEL_PATH = "best_fraud_model_tuned.pkl"
FEATURES_PATH = "feature_names.json"
MAX_ROWS_TO_PROCESS = 50000

# -----------------------------
# Load model and features
# -----------------------------
@st.cache_resource
def load_model():
    if not os.path.exists(MODEL_PATH):
        raise FileNotFoundError(
            f"Model file not found: {MODEL_PATH}. "
            f"Please keep {MODEL_PATH} in the same folder as app.py"
        )
    return joblib.load(MODEL_PATH)

@st.cache_data
def load_feature_names():
    if os.path.exists(FEATURES_PATH):
        with open(FEATURES_PATH, "r", encoding="utf-8") as f:
            return json.load(f)
    return [
        "Time", "V1", "V2", "V3", "V4", "V5", "V6", "V7", "V8", "V9",
        "V10", "V11", "V12", "V13", "V14", "V15", "V16", "V17", "V18",
        "V19", "V20", "V21", "V22", "V23", "V24", "V25", "V26", "V27",
        "V28", "Amount"
    ]

# -----------------------------
# Startup loading
# -----------------------------
try:
    model = load_model()
    expected_features = load_feature_names()
except Exception as e:
    st.error(f"Startup error: {e}")
    st.stop()

if not expected_features:
    st.error("Feature names could not be loaded.")
    st.stop()

# -----------------------------
# Sidebar
# -----------------------------
st.sidebar.title("📌 Model Info")
st.sidebar.write(f"**Model file:** {MODEL_PATH}")
st.sidebar.write("**Prediction target:** Fraud / Non-Fraud")
st.sidebar.write("**Expected feature count:**", len(expected_features))
st.sidebar.write("**Expected columns:**")
st.sidebar.caption(", ".join(expected_features))

threshold = st.sidebar.slider(
    "Prediction Threshold",
    min_value=0.00,
    max_value=1.00,
    value=0.50,
    step=0.01,
    help="Transactions with fraud probability >= threshold will be flagged as fraud."
)

st.sidebar.markdown("---")
st.sidebar.info(
    "Higher threshold = fewer fraud alerts but may miss fraud.\n\n"
    "Lower threshold = more sensitive detection but may increase false alarms.\n\n"
    "⚠️ In fraud detection, lower threshold is often preferred to reduce missed fraud."
)

# -----------------------------
# Main title
# -----------------------------
st.title("💳 Credit Card Fraud Detection Dashboard")
st.write("Upload a CSV file to predict fraudulent transactions and analyze suspicious activity.")

# -----------------------------
# Helper functions
# -----------------------------
def validate_and_prepare_dataframe(df, expected_cols):
    uploaded_cols = list(df.columns)

    missing_cols = [col for col in expected_cols if col not in uploaded_cols]
    extra_cols = [col for col in uploaded_cols if col not in expected_cols]

    if missing_cols:
        return None, False, missing_cols, extra_cols, "missing"

    prepared_df = df[expected_cols].copy()

    for col in expected_cols:
        prepared_df[col] = pd.to_numeric(prepared_df[col], errors="coerce")

    null_counts = prepared_df.isnull().sum()
    bad_cols = null_counts[null_counts > 0]

    if len(bad_cols) > 0:
        return prepared_df, False, bad_cols.to_dict(), extra_cols, "invalid_values"

    return prepared_df, True, [], extra_cols, "ok"

def risk_label(prob):
    if prob >= 0.90:
        return "Very High"
    if prob >= 0.70:
        return "High"
    if prob >= 0.40:
        return "Medium"
    return "Low"

def plot_confusion_matrix_exact(cm):
    fig, ax = plt.subplots(figsize=(6, 5))

    im = ax.imshow(cm)

    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(["Non-Fraud", "Fraud"])
    ax.set_yticklabels(["Non-Fraud", "Fraud"])

    ax.set_xlabel("Predicted Label")
    ax.set_ylabel("True Label")
    ax.set_title("Confusion Matrix")

    threshold_value = cm.max() / 2 if cm.max() > 0 else 0

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(
                j,
                i,
                f"{cm[i, j]}",
                ha="center",
                va="center",
                color="white" if cm[i, j] > threshold_value else "black",
                fontsize=14,
                fontweight="bold"
            )

    fig.colorbar(im)
    plt.tight_layout()
    return fig

# -----------------------------
# File upload
# -----------------------------
uploaded_file = st.file_uploader("Upload CSV file", type=["csv"])

if uploaded_file is not None:
    try:
        raw_df = pd.read_csv(uploaded_file)
    except Exception as e:
        st.error(f"Could not read the CSV file: {e}")
        st.stop()

    if raw_df.empty:
        st.error("The uploaded CSV file is empty.")
        st.stop()

    st.subheader("📄 Uploaded Data Preview")
    st.dataframe(raw_df.head(500), use_container_width=True)

    prepared_df, is_valid, details, extra_cols, status = validate_and_prepare_dataframe(
        raw_df, expected_features
    )

    if extra_cols:
        st.warning(f"Extra columns found and ignored: {extra_cols}")

    if not is_valid:
        if status == "missing":
            st.error("Missing required columns.")
            st.write(details)
        elif status == "invalid_values":
            st.error("Some required columns contain missing or non-numeric values after conversion.")
            st.write(details)
        st.stop()

    if len(prepared_df) > MAX_ROWS_TO_PROCESS:
        st.warning(
            f"Uploaded file contains {len(prepared_df):,} rows. "
            f"Only the first {MAX_ROWS_TO_PROCESS:,} rows will be processed "
            "to keep the app stable on Streamlit Cloud."
        )
        prepared_df = prepared_df.head(MAX_ROWS_TO_PROCESS)
        raw_df = raw_df.head(MAX_ROWS_TO_PROCESS)

    if st.button("🚀 Run Prediction", use_container_width=True):
        try:
            with st.spinner("Running fraud detection model and generating dashboard..."):
                if not hasattr(model, "predict_proba"):
                    raise AttributeError("Loaded model does not support predict_proba().")

                probabilities = model.predict_proba(prepared_df)[:, 1]
                predictions = (probabilities >= threshold).astype(int)

                result_df = raw_df.copy()
                result_df["Fraud_Probability"] = probabilities
                result_df["Prediction"] = predictions
                result_df["Risk_Level"] = [risk_label(p) for p in probabilities]
        except Exception as e:
            st.error(f"Prediction failed: {e}")
            st.stop()

        # -----------------------------
        # Confusion Matrix + Performance
        # -----------------------------
        if "Class" in result_df.columns:
            try:
                y_true = pd.to_numeric(result_df["Class"], errors="coerce")

                if y_true.isnull().any():
                    st.warning("⚠️ Confusion matrix not shown because 'Class' contains missing or non-numeric values.")
                else:
                    y_true = y_true.astype(int)
                    y_pred = result_df["Prediction"].astype(int)

                    valid_labels = set(y_true.unique())
                    if not valid_labels.issubset({0, 1}):
                        st.warning("⚠️ Confusion matrix not shown because 'Class' must contain only 0 and 1.")
                    else:
                        st.subheader("🧩 Confusion Matrix")

                        cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
                        fig = plot_confusion_matrix_exact(cm)
                        st.pyplot(fig)

                        tn, fp, fn, tp = cm.ravel()

                        st.write("### 🔍 Interpretation")
                        st.markdown(f"""
- **True Negatives (TN):** {tn} → Correctly identified non-fraud transactions
- **False Positives (FP):** {fp} → Normal transactions incorrectly flagged as fraud
- **False Negatives (FN):** {fn} → Fraud transactions missed by the model
- **True Positives (TP):** {tp} → Correctly detected fraud transactions
                        """)

                        st.subheader("📈 Model Performance")
                        p_col1, p_col2, p_col3, p_col4 = st.columns(4)
                        p_col1.metric("Accuracy", f"{accuracy_score(y_true, y_pred):.4f}")
                        p_col2.metric("Precision", f"{precision_score(y_true, y_pred, zero_division=0):.4f}")
                        p_col3.metric("Recall", f"{recall_score(y_true, y_pred, zero_division=0):.4f}")
                        p_col4.metric("F1 Score", f"{f1_score(y_true, y_pred, zero_division=0):.4f}")

            except Exception as e:
                st.warning(f"⚠️ Could not generate confusion matrix: {e}")
        else:
            st.info("ℹ️ Confusion matrix not available because uploaded data does not include a 'Class' column.")

        # -----------------------------
        # Top transactions
        # -----------------------------
        st.subheader("🔝 Top 500 Highest-Risk Transactions")
        top_suspicious = result_df.sort_values(
            "Fraud_Probability", ascending=False
        ).head(500)
        st.dataframe(top_suspicious, use_container_width=True)

        # -----------------------------
        # Download predictions
        # -----------------------------
        csv_data = result_df.to_csv(index=False).encode("utf-8")
        st.download_button(
            label="⬇️ Download Predictions",
            data=csv_data,
            file_name="fraud_predictions.csv",
            mime="text/csv",
            use_container_width=True
        )

else:
    st.info("Please upload a CSV file with the required transaction features to begin.")

# -----------------------------
# Footer
# -----------------------------
st.markdown("---")
st.caption(
    "Built with Streamlit for Credit Card Fraud Detection | "
    "Upload transaction data, adjust threshold, review fraud risk, and download predictions."
)

2026-04-16 19:48:04.832 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-16 19:48:04.833 No runtime found, using MemoryCacheStorageManager
2026-04-16 19:48:04.836 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-16 19:48:04.837 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-16 19:48:04.839 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-16 19:48:04.840 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-16 19:48:04.842 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-16 19:48:04.843 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-16 19:48:04.843 Thread 'MainThread':

DeltaGenerator()